[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GeoLarryLai/pyValleyWidth/blob/main/pyValleyWidth_1_CongoCanyon.ipynb)

# pyValleyWidth example 1: Congo Canyon (submarine canyon)

pyValleyWidth was first written to measure mountain valley widths. This notebook applies its revised **`max_valley_width=True`** mode to a bathymetry grid of the Congo Canyon (offshore West Africa, 60 m cells).

In this mode, the single fixed height used by the classic method is replaced by a **rim height that changes along the channel**. Each cross-section is first trimmed to the ridges on either side of the channel, then a "water level" (HAND threshold) is raised step by step while the flooded width is measured. Where the width suddenly jumps (dWidth/dThreshold spikes), the canyon spills over its rim; the **last** such spike between `min_dwdt` (noise floor) and `max_dwdt` (spill-over-the-divide ceiling) sets that cross-section's rim. The per-section rim heights are spread to every channel cell, smoothed with a rolling median, and used to outline the canyon floor, from which widths and depths are measured.

**Data:** `example_data/CongoCanyon/CongoCanyon_bathymetry_smoothed.tif`, a cropped, gap-filled and feature-preserving-smoothed public bathymetry grid (World Mercator, 60 m; values rounded to 0.1 m and compressed for distribution). Figures are written to `figures/`.


In [1]:
# Colab setup: install the package and fetch the example data (skipped when run locally)
import sys
if 'google.colab' in sys.modules:
    !pip -q install pyvalleywidth
    !git clone -q --depth 1 https://github.com/GeoLarryLai/pyValleyWidth.git
    %cd pyValleyWidth


In [2]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1 import make_axes_locatable

import topotoolbox as topo
import pyValleyWidth as pyvw

DATA_DIR = Path('example_data/CongoCanyon')
SMOOTHED_TIF = str(DATA_DIR / 'CongoCanyon_bathymetry_smoothed.tif')

FIG_DIR = Path('figures')      # README figures are saved here
FIG_DIR.mkdir(exist_ok=True)
FIG_DPI = 150

## 1. Load the bathymetry

The shipped grid is already prepared. The preparation steps (crop and downsample the raw grid to 60 m, then fill gaps, fill depressions and apply feature-preserving smoothing with [WhiteboxTools](https://www.whiteboxgeo.com/)) are kept below as an optional, commented-out recipe for your own data.

In [3]:
# Optional DEM preparation recipe (not needed for this example)
# CROP_TIF = "my_dem_cropped.tif"
# FILLED_GAPS_TIF = "my_dem_filled_gaps.tif"
# FILLED_DEP_TIF = "my_dem_filled_depressions.tif"
# import os

# import numpy as np
# import rasterio
# from rasterio.fill import fillnodata

# try:
#     from whitebox.whitebox_tools import WhiteboxTools
# except ImportError as e:
#     raise ImportError(
#         "Install with: pip install whitebox  "
#         "and install the WhiteboxTools binary (see WhiteboxTools install docs)."
#     ) from e

# wbt = WhiteboxTools()
# wbt.set_working_dir(os.getcwd())

# rc = wbt.fill_missing_data(
#     i=CROP_TIF,
#     output=FILLED_GAPS_TIF,
#     filter=11,
#     no_edges=False,
# )
# if rc != 0:
#     raise RuntimeError(f"fill_missing_data failed (exit code {rc}).")

# rc = wbt.fill_depressions(
#     dem=FILLED_GAPS_TIF,
#     output=FILLED_DEP_TIF,
#     fix_flats=True,
#     flat_increment=0.0001,
#     # max_depth=5.0,  # optional: max depression depth to fill (same units as Z)
# )
# if rc != 0:
#     raise RuntimeError(f"fill_depressions failed (exit code {rc}).")





# rc = wbt.feature_preserving_smoothing(
#     # dem=FILLED_NODATA_TIF,
#     dem=FILLED_DEP_TIF,
#     output="my_dem_smoothed.tif",
#     filter=101,  # odd window size (cells)
#     norm_diff=30.0,
#     num_iter=5,
#     max_diff=0.5,
# )
# if rc != 0:
#     raise RuntimeError(f"FeaturePreservingSmoothing failed (exit code {rc}).")

# print(f"Wrote smoothed DEM: {SMOOTHED_TIF}")

In [4]:
dem = topo.read_tif(SMOOTHED_TIF)

fig, ax = plt.subplots(figsize=(20, 8))
dem_norm = colors.Normalize(
    vmin=np.nanmin(np.asarray(dem)), vmax=np.nanmax(np.asarray(dem))
)
dem.plot_hs(ax=ax, cmap='gist_earth', norm=dem_norm)
ax.set_title('Congo Canyon bathymetry (hillshade)')

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='1%', pad=0.05)
elev_sm = ScalarMappable(norm=dem_norm, cmap='gist_earth')
elev_sm.set_array(np.asarray(dem))
fig.colorbar(elev_sm, cax=cax, label='Depth (m)')

plt.tight_layout()
plt.show()

print(f'DEM shape: {dem.shape}')
print(f'Cell size: {dem.cellsize} m')


DEM shape: (1500, 5850)
Cell size: 60.0 m


## 2. Flow Routing and Stream Network


In [5]:
fd = topo.FlowObject(dem)
A = fd.flow_accumulation()

# Keep identical to dem2widths (§3): same area threshold + removeshortstreams(minradius)
streamarea = 5000
minradius = 10
S = topo.StreamObject(fd, threshold=streamarea, units='pixels')
S = pyvw.removeshortstreams(S, minradius)

# How many largest connected catchments to keep? Set None to keep all drainage components on the DEM
klargest_conncomps = 1

# Only keep the largest connected catchment (optional)
if klargest_conncomps is not None:
    S = S.klargestconncomps(int(klargest_conncomps))
S = S.trunk()  # main stem only

fig, ax = plt.subplots(figsize=(20, 8))
dem_array = np.asarray(dem, dtype=float)
vmin_elev = np.nanmin(dem_array)
vmax_elev = np.nanmax(dem_array)
dem_norm = colors.Normalize(vmin=vmin_elev, vmax=vmax_elev)
dem.plot_hs(ax=ax, cmap='gist_earth', norm=dem_norm)
S.plot(ax=ax, color='w', linewidth=0.7, alpha=0.95)
S.plot(ax=ax, color='b', linewidth=1)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='1%', pad=0.05)
elev_sm = ScalarMappable(norm=dem_norm, cmap='gist_earth')
elev_sm.set_array(dem_array)
fig.colorbar(elev_sm, cax=cax, label='Depth (m)')

# Set title according to klargest_conncomps
if klargest_conncomps is None:
    ax.set_title('Stream networks\nBranches (white); main stems (blue)')
else:
    ax.set_title(f'{klargest_conncomps} Largest stream networks\nBranches (white); main stems (blue)')

plt.tight_layout()
fig.savefig(FIG_DIR / 'CongoCanyon_1_map_channel.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()


## 3. Run the Valley Width Pipeline (max_valley_width mode)

This notebook uses the **`max_valley_width=True`** mode of `dem2widths`. Instead of clipping the valley with a single scalar `elevthreshold`, a **variable elevthreshold along the channel** is built up in three simple steps, and then fed into the *same* flood-fill logic used by the classic `max_valley_width=False` mode.

1. **Per-transect spillover detection (width-sensitivity method).** For each cross-profile:
   - The cross-section is clipped to its **local drainage divides** (the position of the maximum HAND value on each side of the channel) so samples beyond a divide are ignored.
   - The HAND threshold is swept upward from near-zero to 95% of the (clipped) profile's maximum HAND, across `n_thr` levels (default 300).
   - At each threshold $T$, the connected valley width is measured by walking outward from the channel on both sides — the edge on each side is the first sample where HAND $\geq T$.
   - The derivative $\text{dWidth}/\text{dThreshold}$ is computed on the (optionally smoothed) width curve. It spikes wherever the valley opens onto a terrace or plateau.
   - All peaks of $\text{dW}/\text{dT}$ are found via `scipy.signal.find_peaks`. Two absolute thresholds (in m/m, the same units as the dW/dT curve) drive the selection: peaks below `min_dwdt` are ignored as noise, and peaks above `max_dwdt` mark a divide-scale breakout (search stops there). The **last** qualifying peak between those bounds is selected as the rim — the spillover point where the valley opens onto the surrounding landscape, before it spills out toward the divide.

2. **Per-transect threshold.** The threshold at the selected spillover peak directly becomes the per-transect valley-depth threshold.

3. **Variable elevthreshold along the channel.** The per-transect thresholds are interpolated onto the **dense stream-node array** (each stream cell inherits the threshold of its nearest profile) and then smoothed along the channel with a **NaN-aware rolling median** of radius `dv_smooth_radius`. The result is a 1-D elevthreshold function along the stream — effectively `allvalleydepth` at every stream node.

The variable threshold is then painted onto the DEM by assigning each pixel the threshold of its nearest **stream node** (not its nearest profile centre), and `DV` is built exactly like in `valleyclass_elev`:

- `is_valley = HAND < threshold_of_nearest_stream_node`
- stream pixels forced to valley
- connected components touching the stream kept; off-channel low-HAND pockets discarded

Because stream nodes are at cellsize spacing, the Voronoi tiles of the threshold raster shrink to a single pixel — the DV boundary is a smooth iso-HAND contour around the channel, just like the constant-threshold flood-fill, only with a threshold that varies slowly along the channel. No hole-filling or morphological closing is needed. Widths are then measured from `DV` with exactly the same `swath_width` procedure as the base mode, so **widths and `DV` are always consistent**.

In this mode `dem2widths` returns a **3-tuple** `(allwidths, DV, allvalleydepth)`. The extra `allvalleydepth` array exposes the per-sample valley depth (m) used for the classification, so the varying-threshold information is preserved for later analysis. In the classic `max_valley_width=False` mode the return is only `(allwidths, DV)` — every transect shares the constant `elevthreshold`, so no per-sample depth array is emitted.

The `dem2widths` function performs all steps (using the **same** `streamarea`, `minradius`, and optional `klargest_conncomps` as §2 so the internal stream network matches `S`):
1. Flow routing and stream extraction
2. Removal of short streams
3. HAND grid (`DZ`) computation (no fixed `elevthreshold`)
4. Cross-stream swath profile generation **on `DZ`**
5. Per-transect spillover rim detection → per-transect depth = last qualifying dW/dT peak between `min_dwdt` and `max_dwdt`
6. Per-transect thresholds lifted to **stream-node resolution** and smoothed along the channel with a rolling median (`dv_smooth_radius`)
7. `DV` built by `HAND < threshold_of_nearest_stream_node` + flood-fill from stream (identical logic to `valleyclass_elev`, just with a variable threshold)
8. Cross-stream swath profile generation **on `DV`** and `swath_width` width extraction
9. Drainage area and gradient extraction at each width point, plus per-sample `allvalleydepth`

### Parameter guide

- **`streamarea`** — Upstream drainage-area threshold used to initiate a stream, in `units='pixels'` (count of upstream cells).
  - **Larger** → fewer, larger streams (trunk-only); cleaner widths, but tributaries disappear.
  - **Smaller** → denser network with small tributaries; more samples, more noise, slower.

- **`max_valley_width`** — Switch between the two valley definitions.
  - **`True`** (this notebook) → variable elevthreshold along the channel; `elevthreshold` is **ignored**.
  - **`False`** → classic mode using a single scalar `elevthreshold`.

- **`min_dwdt`** — (**m/m**) Absolute floor on dW/dT for a peak to qualify. Peaks below this value are treated as noise and ignored. Default **0.0** (accept every detected peak).
  - **Higher** (e.g. `5.0`) → stricter; only large spillover spikes count as rim candidates.
  - **`0.0`** → most permissive; relies on `dw_smooth` and `max_dwdt` for noise control.
  - Read a value directly off the dW/dT curve in the diagnostic cell (§3.1) and use it here.

- **`max_dwdt`** — (**m/m**) Absolute ceiling on dW/dT. Any peak exceeding this is treated as a divide-scale breakout: the search stops there and the last qualifying peak before that point is chosen as the rim. Default **`None`** (no ceiling).
  - Set this to clip out the huge late spike that occurs when the valley spills onto the divide. Pick a number a bit below that spike but well above your real valley-rim spikes.
  - **`None`** → no breakout cap; the last qualifying peak anywhere in the sweep is the rim.

- **`clip_to_divides`** — Default **`True`**. When True, every transect is clipped to its local drainage divides before the sweep. Set to `False` only if you want the raw transect (extends through neighbouring drainages) to be analysed.

- **`n_thr`** — Number of threshold levels in the width-sensitivity sweep. Default **300**. You typically don't need to change this; 300 is plenty.

- **`dw_smooth`** — Uniform-filter window (in sweep samples) for smoothing the width curve before differentiation. Default **5**. A small reasonable smoother — set higher to broaden peaks, lower (or `1`) to disable.

- **`profile_smooth_radius`** — (**m**) 1-D smoothing radius applied to each HAND profile before the sweep. Default **0.0** (off). Set to a few cellsizes to suppress micro-noise on rough DEMs.

- **`dv_smooth_radius`** — (**m**) Rolling-median window radius applied along the channel to the per-stream-node threshold array. Default = `minradius`.
  - **Larger** → smoother `allvalleydepth` along the channel, DV boundary becomes smoother and more uniform between tributary-scale variations.
  - **Smaller** / `0` → threshold follows each individual profile; fastest response to real depth changes, but residual per-profile noise is more visible.

- **`swath_dx`** — Along-stream spacing (**m**) between cross-stream profiles.
  - **Larger** → fewer profiles, faster.
  - **Smaller** → denser along-stream sampling, slower.

- **`minradius`** — (**m**) Two roles: (1) streams shorter than this are removed; (2) moving-minimum smoothing radius applied to the raw widths.
  - **Larger** → more tributaries pruned; strongly smoothed widths (narrow pinch-points get lost).
  - **Smaller** → keeps short tributaries and local width variability; noisier widths.

- **`swath_width_param`** — Total cross-stream transect length (**m**). Should comfortably exceed the distance from channel to divide on each side; the divide-clipping step then trims it down. **`'auto'`** in `max_valley_width=True` → DEM cap = half of the DEM's shorter side (rounded to an even cellsize multiple).

- **`main_trunk_only`** —
  - **True** → measure widths only along the main trunk (`S.trunk()`); matches the single-line map view.
  - **False** → measure along every stream in `S`, including tributaries.

- **`n_jobs`** — Number of parallel workers for per-transect spillover detection (`max_valley_width=True` only). Default **-1** (all CPU cores). Set to `1` for serial execution.

- **`nodata_value`** — How "empty" DEM pixels are handled before flow routing and HAND computation.
  - **`'auto'`** (recommended) → detect the DEM's nodata sentinel and always mask NaN/Inf.
  - **A number** → force that sentinel value to be treated as nodata.
  - **`None`** → disable masking entirely; only safe on gap-free DEMs.

- **`drop_saturated`** — Post-filter on `allwidths`/`allvalleydepth` using the `saturated` flag (column 7).
  - **True** → discard samples where at least one bank failed to produce a rim (transect too short / edge artefacts).
  - **False** → keep every sample.


### How HAND is computed

**HAND** (Height Above Nearest Drainage) is the vertical elevation difference between each DEM pixel and the stream cell it drains to. It is computed by `FlowObject.vertdistance2stream(StreamObject, DEM)`:

1. For every pixel in the DEM, trace the steepest-descent flow path (from `FlowObject`) downhill until a **stream cell** (from `StreamObject`) is reached.
2. Record the elevation difference: $\text{HAND}(p) = z(p) - z(s)$, where $s$ is the stream cell that $p$ drains into.

Pixels *on* the stream have HAND = 0. Moving laterally away from the channel, HAND increases as the terrain rises toward the valley walls and ridges.

### How the spillover rim is detected

Each cross-stream transect cuts the valley perpendicularly. Before any analysis, the transect is **clipped to its local drainage divides** — the position of the maximum HAND value on each side of the channel — so samples beyond a divide (which belong to a neighbouring drainage) are ignored. The algorithm then sweeps HAND thresholds upward across the clipped cross-section (both banks at once) and measures the connected valley width at each level. As the threshold rises, the width grows — and where it **jumps** (a terrace or the valley rim opens up), the derivative $\text{dW}/\text{dT}$ spikes. Two absolute thresholds (both in m/m, the same units as the dW/dT curve) control which peaks count: `min_dwdt` is a noise floor (peaks below it are ignored) and `max_dwdt` is a breakout ceiling (any peak above it marks the moment the valley spills onto the divide, so the search stops there). The **last** qualifying peak between those bounds is selected as the rim — the spillover point where the valley opens onto the surrounding landscape, not an inner terrace and not the divide itself.

The cell below extracts 6 example cross-sections and visualises the spillover rim detection step by step.


In [6]:
# Diagnostic: 6 example cross-sections using plot_transect_diagnostic.
# Each call produces a clean three-panel figure (map + HAND profile +
# width-sensitivity sweep) for one trunk-channel location.

# Two user-facing knobs (must match the dem2widths call below).
# Both are absolute thresholds in dW/dT units (m/m), the same units as
# the red dW/dT curve in the bottom-right panel.
_min_dwdt = 10.0     # floor: peaks below this are noise, ignored
_max_dwdt = 100    # ceiling: peaks above this mark a divide-scale breakout (None = off)

# Save one representative cross-section for the README (1-6)

# Pick 6 trunk-node points whose x-coordinates are evenly spread along the
# easting range of the trunk (avoids clustering when the channel zig-zags).
_S_trunk = S.trunk()
_rows, _cols = _S_trunk.node_indices
_t = _S_trunk.transform
_xs = _t[0] * _cols + _t[2]
_ys = _t[4] * _rows + _t[5]
_N = len(_xs)
_x_targets = np.linspace(_xs.min(), _xs.max(), 8)[1:-1]   # 6 interior targets
_sel = sorted({int(np.argmin(np.abs(_xs - _xt))) for _xt in _x_targets})
_pc = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#a65628']
print(f'{_N} trunk nodes; rendering {len(_sel)} transects at indices {_sel}')

for k, idx in enumerate(_sel):
    _xy = (float(_xs[idx]), float(_ys[idx]))
    pyvw.plot_transect_diagnostic(
        dem, fd, _S_trunk, _xy,
        min_dwdt=_min_dwdt,
        max_dwdt=_max_dwdt,
        title=f'Profile {k+1}',
        color=_pc[k],
        dpi=FIG_DPI,
    )
    plt.show()


7430 trunk nodes; rendering 6 transects at indices [85, 1780, 2676, 3407, 5860, 6844]


In [7]:
# Parameters (streamarea & minradius must match the stream network in §2)
streamarea = 5000             # upstream drainage-area threshold for stream initiation (units='pixels' -> count of cells). larger = sparser trunk-only net; smaller = denser net w/ more tributaries
swath_dx = 50                 # along-stream spacing between cross-stream profiles (m). smaller = denser samples & slower; larger = fewer samples & faster
minradius = 30000             # (m) two roles: (1) min stream length kept; (2) moving-min smoothing radius for widths. larger = fewer tribs & smoother widths; smaller = more detail & more noise
swath_width_param = 'auto'    # total cross-stream transect length (m). 'auto' (max_valley_width=True) = DEM-cap (half of shorter side); number = force. too small -> saturated samples; too large -> slower
main_trunk_only = True        # True: widths only along S.trunk() (single main stem); False: every stream in S (all tributaries)
nodata_value = 'auto'         # 'auto' = detect DEM sentinel + always mask NaN/Inf; number = force that sentinel; None = disable masking entirely

# --- max_valley_width mode (spillover / width-sensitivity) ---
# True: variable elevthreshold along the channel. Each transect is clipped to its
# local drainage divides, the HAND threshold is swept upward, and the per-transect
# rim is the last qualifying dW/dT peak between min_dwdt and max_dwdt. Thresholds
# are lifted to stream-node resolution, smoothed along the channel with a rolling
# median, and consumed by the same flood-fill logic as max_valley_width=False.
max_valley_width = True

# Two user-facing knobs for spillover-rim detection. Both are absolute thresholds
# on dW/dT (m/m); read values off the diagnostic dW/dT plot in §3.1 and use them here.
min_dwdt = 10                  # floor: peaks below this are noise, ignored. raise (e.g. 5-10) to suppress small bumps
max_dwdt = 100                 # ceiling: peaks above this mark a divide-scale breakout. None disables the cap

dv_smooth_radius = minradius    # (m) rolling-median window applied along the channel to the per-stream-node threshold array. larger = smoother along-channel elevthreshold
n_jobs = -1                     # parallel workers for spillover detection. -1 = all cores; 1 = serial

allwidths, DV, allvalleydepth = pyvw.dem2widths(
    dem,
    streamarea=streamarea,
    swath_dx=swath_dx,
    minradius=minradius,
    swath_width_param=swath_width_param,
    units='pixels',
    plot=True,
    klargest_conncomps=klargest_conncomps,
    main_trunk_only=main_trunk_only,
    nodata_value=nodata_value,
    max_valley_width=max_valley_width,
    min_dwdt=min_dwdt,
    max_dwdt=max_dwdt,
    dv_smooth_radius=dv_smooth_radius,
    n_jobs=n_jobs,
)

print(f'\nOutput shape: {allwidths.shape}')
print(f'Columns: x, y, raw_width, min_width, drainage_area_m2, gradient_deg, saturated')
print(f'Number of width measurements: {allwidths.shape[0]}')

if allvalleydepth.size > 0:
    finite_depth = allvalleydepth[np.isfinite(allvalleydepth)]
    if finite_depth.size:
        print(
            f'allvalleydepth (m): min={finite_depth.min():.2f}, '
            f'median={np.median(finite_depth):.2f}, '
            f'mean={finite_depth.mean():.2f}, '
            f'max={finite_depth.max():.2f} '
            f'(NaN entries: {int((~np.isfinite(allvalleydepth)).sum())})'
        )


No finite nodata sentinel attribute found on DEM; masking only non-finite (NaN/Inf) pixels.
Calculating flow directions


Calculating streams
Removing short streams


Restricting to 1 largest connected stream component(s)
Restricting stream network to map-style main trunk (StreamObject.trunk)
swath_width_param='auto' (max_valley_width=True): using DEM cap = 45000 m.
Computing HAND grid (DZ) for max_valley_width mode


Swath sampling on HAND (for rim detection)


Per-transect spillover rim detection (min_dwdt=10, max_dwdt=100, n_thr=300)


Rolling-median smoothing of along-channel threshold (radius=30000 m) applied at stream-node resolution
Building DV from variable along-channel elevthreshold


Swath sampling on DV (for width extraction)


Extracting drainage area
Extracting gradients
Flagged 264 saturated width sample(s) (transect never reached a hillslope on at least one side).
Done. 10445 width measurements extracted.

Output shape: (10445, 7)
Columns: x, y, raw_width, min_width, drainage_area_m2, gradient_deg, saturated
Number of width measurements: 10445
allvalleydepth (m): min=1.48, median=208.99, mean=470.20, max=2730.71 (NaN entries: 0)


## 4. Canyon floor outline

In [8]:
fig, ax = plt.subplots(figsize=(20, 8))

# Hillshade background
dem.plot_hs(ax=ax, cmap='gray',
            norm=colors.Normalize(vmin=np.nanmin(np.asarray(dem)), vmax=np.nanmax(np.asarray(dem))))

# Valley polygon overlay: use cyan color and alpha 0.5
dv_arr = np.asarray(DV.z).copy()
bounds = dem.bounds
extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

# Only plot valleys: mask everything that isn't class 1 (valley)
valley_mask = np.ma.masked_where(dv_arr != 1, dv_arr)
im = ax.imshow(valley_mask, extent=extent, origin='upper',
               cmap=colors.ListedColormap(['cyan']), alpha=0.5, vmin=1, vmax=1.1)

ax.set_title('Canyon floor outline (cyan)')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()
fig.savefig(FIG_DIR / 'CongoCanyon_2_canyon_outline.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()


## 5. Visualize valley widths (raw and stream-smoothed)

The first map shows **raw** swath-derived width at each sample. The second map assigns each sample to the nearest vertex on the stream network from §2 (`S`, largest catchment when filtering is on), averages where several samples hit the same vertex, gap-fills along the network, then applies the same Gaussian smoothing along streams as for $k_{sn}$ (see `pyValleyWidth.smooth_valley_width_on_stream_network`).


In [9]:
# Two full-sized maps: raw widths, then stream-network smoothed widths
vw_match_dist = 10   # m — max distance to snap a width sample to a stream vertex
vw_smooth_window = 100

vw_smooth_mapped, _dvw, _ivw, n_matched = pyvw.smooth_valley_width_on_stream_network(
    allwidths, S, max_match_dist=vw_match_dist, window_size=vw_smooth_window, width_column=2
)
print(f'Stream-smoothed valley width: matched samples = {n_matched} / {len(allwidths)}')

# Stream-smoothed width
fig, ax = plt.subplots(figsize=(20, 8))
dem.plot_hs(ax=ax, cmap='gray',
            norm=colors.Normalize(vmin=np.nanmin(np.asarray(dem)), vmax=np.nanmax(np.asarray(dem))))
mask_plot = np.isfinite(vw_smooth_mapped)

sc = ax.scatter(allwidths[mask_plot, 0], allwidths[mask_plot, 1],
                c=vw_smooth_mapped[mask_plot], cmap='inferno', s=5,
                # vmin=940, vmax=1020
                )
plt.colorbar(sc, ax=ax, label='Stream-smoothed Valley Width (m)', shrink=0.6)
ax.set_title(f'Stream-smoothed Valley Width (snap<{vw_match_dist} m, window={vw_smooth_window})')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()
fig.savefig(FIG_DIR / 'CongoCanyon_3_canyon_width.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()


Stream-smoothed valley width: matched samples = 2951 / 10445


In [10]:
# Per-sample valley depth (allvalleydepth): raw, then stream-network smoothed
vd_match_dist = vw_match_dist
vd_smooth_window = vw_smooth_window

# Pack [x, y, depth] so we can reuse the same stream-network smoother
depth_xyz = np.column_stack([allwidths[:, 0], allwidths[:, 1], allvalleydepth])
vd_smooth_mapped, _dvd, _ivd, n_matched_d = pyvw.smooth_valley_width_on_stream_network(
    depth_xyz, S, max_match_dist=vd_match_dist, window_size=vd_smooth_window, width_column=2
)
print(f'Stream-smoothed valley depth: matched samples = {n_matched_d} / {len(allvalleydepth)}')

# Stream-smoothed depth
fig, ax = plt.subplots(figsize=(20, 8))
dem.plot_hs(ax=ax, cmap='gray',
            norm=colors.Normalize(vmin=np.nanmin(np.asarray(dem)), vmax=np.nanmax(np.asarray(dem))))
mask_plot = np.isfinite(vd_smooth_mapped)
sc = ax.scatter(allwidths[mask_plot, 0], allwidths[mask_plot, 1],
                c=vd_smooth_mapped[mask_plot], cmap='BuPu_r', s=5,
                # vmin=0, vmax=50
                )
plt.colorbar(sc, ax=ax, label='Stream-smoothed Valley Depth (m)', shrink=0.6)
ax.set_title(f'Stream-smoothed Valley Depth (snap<{vd_match_dist} m, window={vd_smooth_window})')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()
fig.savefig(FIG_DIR / 'CongoCanyon_5_canyon_depth.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()


Stream-smoothed valley depth: matched samples = 2951 / 10445


## 6. $k_{\mathrm{sn}}$ analysis

A small minimum gradient is imposed on the DEM so channel gradients are well-behaved. A slope–area regression (optional diagnostic figure) estimates reference statistics; $k_{\mathrm{sn}}$ is then computed with a **fixed** concavity $\theta$ suggested by the S-A plot. We map **raw** $k_{\mathrm{sn}}$, then **smooth** values along the stream network.


In [11]:
from topotoolbox.stream_functions import imposemin

acc = A
s2 = S.klargestconncomps(1)

dem_imposed = imposemin(fd, dem, minimum_slope=0.000001)

print('Slope–area (reference statistics; shows diagnostic figure if plot_fit=True):')
STATS = pyvw.slopearea(s2, dem_imposed, acc, plot_fit=True)
print(f"  theta (fitted): {STATS['theta']:.4f}, ks: {STATS['ks']:.4f}, R^2: {STATS['r_squared']:.4f}")

Slope–area (reference statistics; shows diagnostic figure if plot_fit=True):
  theta (fitted): 0.2844, ks: 0.0000, R^2: 0.0569


In [12]:
theta_fixed = -0.3 # based on the S-A plot regression theta value

ksn_raw = pyvw.calculate_ksn(s2, dem_imposed, acc, theta_fixed)
print(f'k_sn raw range: {np.nanmin(ksn_raw):.2f} … {np.nanmax(ksn_raw):.2f}')

ksn_smoothed = pyvw.smooth_stream_values(s2, ksn_raw, window_size=100)
pyvw.plot_stream_ksn(s2, dem, ksn_smoothed, vmin=0,
vmax=15, 
figsize=(20, 5),
title='Normalized channel steepness (k_sn) — smoothed')


k_sn raw range: 0.00 … 216.40


## 7. Long profile and χ–Z profile (main channel in color, branches gray)

The main stem is `st = s2.trunk()`. Gray points are all channel nodes in `s2`; the main channel is colored by smoothed $k_{\mathrm{sn}}$ with fixed color limits (`ksn_plot_vmax`).

In [13]:
st = s2.trunk()  
ksn_st = pyvw.calculate_ksn(st, dem_imposed, acc, theta_fixed)
ksn_st_smoothed = pyvw.smooth_stream_values(st, ksn_st, window_size=200)

# Fixed color limits 0 … ksn_plot_vmax (None = autoscale to trunk max)
ksn_plot_vmax = 15

# pyvw.plot_longitudinal_trunk_colored_ksn(
#     s2, st, dem, ksn_st_smoothed, vmin=0, 
#     vmax=ksn_plot_vmax, 
#     theta=theta_fixed,
# )
pyvw.plot_chi_z_trunk_colored_gray_network(
    s2, st, dem, acc, ksn_st_smoothed, vmin=0, 
    vmax=ksn_plot_vmax, 
    theta=theta_fixed,
)


## 8. Long profile with canyon width and canyon-top depth (twin axis)

Gray distance–depth points for every `s2` node, and the main channel colored by **smoothed** $k_{\mathrm{sn}}$.

The **valley-width** overlay on the twin axis now has three layers:

1. **Stream-network smoothed dots** — the per-transect widths are smoothed on `s2` (window `valleywidth_window`, in stream nodes) then sampled onto trunk NAL vertices. Plotted as faint steelblue dots (`alpha=0.2`), one per trunk vertex along distance.
2. **Distance-based moving-average curve** — a NaN-aware moving mean of those smoothed dots using window `ma_window_m` (in **meters** along trunk distance). Drawn as a solid steelblue line.
3. **Shading** — `fill_between(0, MA)` at `alpha=0.25` under the MA curve only.

When `allvalleydepth` is supplied (only available with `max_valley_width=True`), the same three-layer treatment is drawn on the **elevation** axis for the **valley top** (`valley_top_z = channel_z + smoothed_depth`):

1. Red dots (`alpha=0.2`) at each trunk vertex for the smoothed valley-top elevation.
2. Red MA curve (`alpha=0.6`, window `ma_window_m`).
3. Red shading (`alpha=0.3`) between the channel profile and the MA valley-top curve.

**Axis units.** Distance, width, and elevation axes default to meters. When the max magnitude on an axis exceeds 10 000, values are divided by 1000 and the label becomes `(x1000 m)` (functionally kilometres, just labeled explicitly so the underlying unit stays clear).

### Parameter guide

- **`theta`** — Reference concavity used by `calculate_ksn` ($k_{sn} = S \cdot A^{-\theta}$). Convention here is **negative** (`-0.45` is typical for bedrock rivers).
  - **More negative** (e.g. `-0.55`) → flattens $k_{sn}$ contrast in steep headwaters.
  - **Less negative** (e.g. `-0.35`) → amplifies $k_{sn}$ contrast.
  - Should match the value used elsewhere in the notebook (`theta_fixed`).

- **`ksn_window`** — Gaussian smoothing window (in **stream nodes**) applied to per-node $k_{sn}$ along the trunk via `smooth_stream_values`.
  - **Larger** → cleaner, longer-wavelength $k_{sn}$ trends; reach-scale knickpoints survive, sub-reach noise gone.
  - **Smaller** → preserves local $k_{sn}$ spikes at the cost of more visual scatter.

- **`valleywidth_window`** — Gaussian smoothing window (in **stream nodes**) for the **valley-width** field on `s2` (renamed from `vw_window`). Controls how smooth the dots are.
  - **Larger** → smooth, slowly-varying width dots; tight pinches are washed out.
  - **Smaller** → keeps narrow constrictions, more jitter from per-transect noise.

- **`valleydepth_window`** — Optional separate Gaussian smoothing window for the **valley-depth** field on `s2` (renamed from `depth_window`; only used when `allvalleydepth` is provided).
  - **`None`** (default) → reuse `valleywidth_window`.
  - **A number** → smooth depth more or less than width. Useful when widths are noisier than depths (or vice versa).

- **`ma_window_m`** — (**m**) Shared **distance-based** moving-average window used for the width curve AND the valley-top curve. Acts on top of the `s2`-smoothed dots and drives the shaded regions.
  - **Larger** → slow, reach-scale trends; short local variability averaged away.
  - **Smaller** → keeps local structure; curve sticks closer to the dots.
  - Differs from `valleywidth_window` / `valleydepth_window` by operating on **distance** (meters) rather than on **stream-node counts** on the `s2` graph.

- **`max_match_dist`** — (**m**) Maximum distance for snapping each width/depth sample to the nearest `s2` vertex. Samples farther than this are dropped from the overlay.
  - **Smaller** → stricter snap; off-network samples (saturated edges, side-branches) excluded.
  - **Larger** → more samples retained, but risk of mis-attaching far-away samples to the wrong reach.

- **`vw_outline_sigma_pts`** — **Deprecated**. Retained in the signature for API stability but no longer used by the curve (the moving average replaces the old outline smoothing). Any value is accepted and ignored.

- **`ksn_clim`** — `(vmin, vmax)` for the $k_{sn}$ colormap on the trunk scatter. Overrides `ksn_vmin`/`ksn_vmax`.
  - **`None`** → auto-scale (`0` → data max).
  - **Fixed tuple** (e.g. `(0, 300)`) → comparable colors across notebooks/sites.

- **`valleywidth_ylim`** — `(ymin, ymax)` for the **twin** valley-width axis on the right, given in **meters** (the limits are internally scaled to match the axis unit when `(x1000 m)` is active).
  - **`None`** → auto.
  - **Fixed tuple** → useful when comparing reaches with very different width ranges, or hiding outliers.

- **`allvalleydepth`** — Per-sample valley depth from `dem2widths(..., max_valley_width=True)`, aligned 1:1 with `allwidths`.
  - **`None`** (default) → no red valley-top overlay drawn.
  - **Provide it** → enables the red smoothed dots + MA curve + shading on the elevation axis.

- **`figsize`** — `(width, height)` of the figure in inches.

- **`savefig`** — Path to save the figure (PNG/PDF). `None` skips saving.


In [14]:
pyvw.plot_longitudinal_trunk_ksn_valley_width_twin(
    s2, st, dem, dem_imposed, acc, allwidths,
    theta=theta_fixed,                 # reference concavity (negative); should match the value used elsewhere
    ksn_window=200,                     # Gaussian window (stream nodes) smoothing k_sn along the trunk
    valleywidth_window=5,             # Gaussian window (stream nodes) smoothing valley width on s2
    valleydepth_window=5,             # Gaussian window (stream nodes) smoothing valley depth on s2; None -> reuse valleywidth_window
    ma_window_m=10000.0,                 # (m) distance-based moving-average window used for BOTH width and valley-top curves / shading
    max_match_dist=50.0,               # (m) max snap distance from a width/depth sample to s2 vertex
    vw_outline_sigma_pts=0.1,          # (deprecated) kept for API stability; no longer used by the curve
    ksn_clim=(0, 15),                 # (vmin, vmax) for the k_sn colormap; None -> auto (0 .. data max)
    valleywidth_ylim=(1000, 25000),    # (ymin, ymax) for the twin valley-width axis; None -> auto
    allvalleydepth=allvalleydepth,     # enables red valley-top overlay (smoothed dots + MA curve + shading); None -> off
    savefig=str(FIG_DIR / 'CongoCanyon_4_long_profile.png'),
)

Valley width points matched to s2 network: 10445 / 10445


## 9. Threshold and excess topography (5° slope)

**Threshold topography:** `GridObject.excesstopography(method='fsm2d', threshold=tan(5°))`. **Excess topography:** difference between the original DEM and that threshold surface, shown for the **full** extent (no catchment mask).


In [15]:
pyvw.plot_threshold_and_excess_topography(dem, slope_degrees=5.0, basin_mask=None, figsize=(20, 8))